In [1]:
import pandas as pd
import cv2
import numpy as np
from pathlib import Path
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

from src.blank_detect import is_blank

In [2]:
BASE_DIR = Path.cwd()
CSV_PATH = BASE_DIR / "data" / "output" / "dataset" / "labels.csv"
IMG_DIR = BASE_DIR / "data" / "output" / "dataset" / "images"

In [3]:
df = pd.read_csv(CSV_PATH)

images = []
labels = df["is_blank"].values

In [4]:
for filename in df["filename"]:
    img_path = str(IMG_DIR / filename)
    
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE) 
    
    if img is None:
        print(f"Warning: Could not read {img_path}")
        continue
        
    images.append(img)

In [5]:
X_data = np.array(images)
y_data = labels

print(f"Images array shape: {X_data.shape}")
print(f"Labels array shape: {y_data.shape}")

Images array shape: (500, 500, 500)
Labels array shape: (500,)


In [6]:
df['detected_blank'] = [
    int(is_blank(cv2.imread(str(IMG_DIR / f)), lower_threshold=0.005)) 
    for f in df['filename']
]

df['modified_y'] = (~df['content_type'].isin(['mixed', 'text_only'])).astype(int)

In [8]:
import numpy as np
import cv2
from sklearn.ensemble import RandomForestClassifier
from strategies.preprocessor import BlankDetector
import joblib
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_data, 
    y_data, 
    test_size=0.20, 
    random_state=42, 
    stratify=y_data
)

print(f"Train set: {X_train.shape}, {y_train.shape}")
print(f"Test set:  {X_test.shape}, {y_test.shape}")

print(f"Training blank ratio: {y_train.mean():.2%}")
print(f"Testing blank ratio:  {y_test.mean():.2%}")

print("Extracting features from training set...")
X_train_features = np.array([BlankDetector.extract_blank_features(img) for img in X_train])

print("Extracting features from test set...")
X_test_features = np.array([BlankDetector.extract_blank_features(img) for img in X_test])

print(f"New X_train shape: {X_train_features.shape}")
print(f"New X_test shape:  {X_test_features.shape}")

print(f"Old X_train shape: {X_train_features.shape}") # Sẽ in ra (N, 1, 3)

# ---------------------------------------------------------
# BƯỚC ÉP PHẲNG (SQUEEZE / RESHAPE) TỪ 3D XUỐNG 2D
# ---------------------------------------------------------
X_train_features = X_train_features.reshape(X_train_features.shape[0], 3)
X_test_features = X_test_features.reshape(X_test_features.shape[0], 3)

print(f"Fixed X_train shape: {X_train_features.shape}") # Sẽ in ra (N, 3) -> Chuẩn!

# Tiến hành Train
clf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
clf.fit(X_train_features, y_train)

# Lưu Model ngay lập tức
joblib.dump(clf, "models/rf_blank_detector.joblib")
print("Training complete & Model saved!")

Train set: (400, 500, 500), (400,)
Test set:  (100, 500, 500), (100,)
Training blank ratio: 21.00%
Testing blank ratio:  21.00%
Extracting features from training set...
Extracting features from test set...
New X_train shape: (400, 1, 3)
New X_test shape:  (100, 1, 3)
Old X_train shape: (400, 1, 3)
Fixed X_train shape: (400, 3)
Training complete & Model saved!


In [9]:
from sklearn.metrics import classification_report


y_pred = clf.predict(X_test_features)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        79
           1       1.00      1.00      1.00        21

    accuracy                           1.00       100
   macro avg       1.00      1.00      1.00       100
weighted avg       1.00      1.00      1.00       100



In [ ]:
test_image_dir = Path("test/dataset/images")
test_csv_path = Path("test/dataset/labels.csv")

df_test = pd.read_csv(test_csv_path)

test_images = []
labels = df_test["is_blank"].values

for filename in df_test["filename"]:
    img_path = str(test_image_dir / filename)
    
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE) 
    
    if img is None:
        print(f"Warning: Could not read {img_path}")
        continue
        
    test_images.append(img)

X_test_data = np.array(test_images)
y_test_data = labels

X_test_features = np.array([BlankDetector.extract_blank_features(img) for img in X_test_data])
X_test_features = X_test_features.reshape(X_test_features.shape[0], 3)
print(f"Images array shape: {X_test_data.shape}")
print(f"Labels array shape: {y_test_data.shape}")

Images array shape: (5000, 500, 500)
Labels array shape: (5000,)


In [13]:
y_pred = clf.predict(X_test_features)
print(classification_report(y_test_data, y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00      3990
           1       1.00      1.00      1.00      1010

    accuracy                           1.00      5000
   macro avg       1.00      1.00      1.00      5000
weighted avg       1.00      1.00      1.00      5000

